In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import copy
import pickle


class CVAE(nn.Module):
  def __init__(self):
    super(CVAE, self).__init__()


    self.fc1 = nn.Linear(9, 512)  # 7 columns + 2 conditions
    self.fc2 = nn.Linear(512, 256)
    self.fc3 = nn.Linear(256, 128)
    self.fc21 = nn.Linear(128, 30)
    self.fc22 = nn.Linear(128, 30)
    self.dropout = nn.Dropout(p=0.2)


    self.fc4 = nn.Linear(33, 128)  # 30 latent space + 2 conditions
    self.fc5 = nn.Linear(128, 256)
    self.fc6 = nn.Linear(256, 512)
    self.fc7 = nn.Linear(512, 6)

  def encode(self, x, condition):
    combined = torch.cat([x, condition], 1)
    h1 = F.relu(self.fc1(combined))
    h2 = F.relu(self.fc2(self.dropout(h1)))
    h3 = F.relu(self.fc3(self.dropout(h2)))
    return self.fc21(h3), self.fc22(h3)

  def reparameterize(self, mu, logvar):
    std = torch.exp(0.5*logvar)
    eps = torch.randn_like(std)
    return mu + eps*std

  def decode(self, z, condition):
    combined = torch.cat([z, condition], 1)
    h4 = F.relu(self.fc4(combined))
    h5 = F.relu(self.fc5(self.dropout(h4)))
    h6 = F.relu(self.fc6(self.dropout(h5)))
    return self.fc7(self.dropout(h6))

  def forward(self, x, condition):
    mu, logvar = self.encode(x, condition)
    z = self.reparameterize(mu, logvar)
    return self.decode(z, condition), mu, logvar

def cvae_loss(recon_x, x, mu, logvar, beta=0.001):
  MSE = F.mse_loss(recon_x, x, reduction='sum')
  KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
  base = MSE + beta*KLD

  return base

def train_cvae(cvae, train_loader, val_loader, epochs=300, patience=50, beta=0.001, lr=1e-3, device="cuda"):

  cvae.to(device)


  optimizer = optim.AdamW(cvae.parameters(), lr=lr)
  scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.7)

  best_val_loss = float("inf")
  best_state = None
  no_improve_epochs = 0

  train_losses = []
  val_losses = []


  for epoch in range(epochs):
    cvae.train()
    train_total = 0.0

    for y, x in train_loader:
      y = y.to(device)
      x = x.to(device)

      optimizer.zero_grad()

      recon_batch, mu, logvar = cvae(x, y)
      loss = cvae_loss(
                recon_batch, x, mu, logvar,
                beta=beta
            )

      loss.backward()
      optimizer.step()

      train_total += loss.item()


    avg_train_loss = train_total / len(train_loader.dataset)
    train_losses.append(avg_train_loss)

    cvae.eval()
    val_total = 0.0

    with torch.no_grad():
      for y, x in val_loader:
        y = y.to(device)
        x = x.to(device)

        recon_batch, mu, logvar = cvae(x, y)
        val_loss = cvae_loss(
                        recon_batch, x, mu, logvar,
                        beta=beta
                    )


        val_total += val_loss.item()

    avg_val_loss = val_total / len(val_loader.dataset)
    val_losses.append(avg_val_loss)

    print(
                f"[CVAE-Base] Epoch {epoch+1}, "
                f"Train Total: {avg_train_loss:.4f}, "
                f"Val Total: {avg_val_loss:.4f}"
            )

    if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      best_state = copy.deepcopy(cvae.state_dict())
      no_improve_epochs = 0
    else:
      no_improve_epochs += 1
      if no_improve_epochs >= patience:
        print(
                        f"Early stopping at epoch {epoch+1}. "
                        f"No improvement in validation loss for {patience} consecutive epochs."
                    )
        break
    scheduler.step()

  if best_state is not None:
    cvae.load_state_dict(best_state)
    print(f"Loaded best CVAE base model with Val Loss: {best_val_loss:.4f}")
  return cvae

def generate_outputs(model, input_data, num_samples=10):
  model.eval()
  device = next(model.parameters()).device  # get model device

  with torch.no_grad():
    input_df = pd.DataFrame(
    input_data,columns=['Frequency (Hz)', 'Storage modulus (Pa)', 'Loss modulus (Pa)'])
    input_data_scaled = Y_scaler.transform(input_df)

    conditions = torch.tensor(input_data_scaled, dtype=torch.float32, device=device)

    outputs = []
    for _ in range(num_samples):
      z = torch.randn(conditions.size(0), 30, device=device)
      output = model.decode(z, conditions)
      output = X_scaler.inverse_transform(output.cpu().numpy())
      outputs.append(output)

  return outputs

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_default_dtype(torch.float32)


device = 'cuda' if torch.cuda.is_available() else "cpu"

df_train = pd.read_csv('/content/train.csv')
df_val = pd.read_csv('/content/val.csv')


X_train, X_val = df_train.iloc[:, 0:6], df_val.iloc[:, 0:6]
y_train, y_val = df_train.iloc[:, 6:9], df_val.iloc[:, 6:9]

X_scaler = StandardScaler()
Y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
y_train_scaled = Y_scaler.fit_transform(y_train)

with open('/content/X_scaler_cvae.pkl', 'wb') as f:
  pickle.dump(X_scaler, f)
with open('/content/Y_scaler_cvae.pkl', 'wb') as f:
  pickle.dump(Y_scaler, f)

X_val_scaled = X_scaler.transform(X_val)
y_val_scaled = Y_scaler.transform(y_val)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

train_data = TensorDataset(y_train_tensor, X_train_tensor)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)

val_data = TensorDataset(y_val_tensor, X_val_tensor)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)


cvae = CVAE()
best_cvae = train_cvae(cvae, train_loader, val_loader, epochs = 300, patience= 20, beta=0.001, device=device)


df_test = pd.read_csv("/content/test.csv")

frequency, storage_modulus, loss_modulus = df_test['Frequency (Hz)'], df_test['Storage modulus (Pa)'], df_test['Loss modulus (Pa)']


# Initialize an empty DataFrame with specified columns
columns = ['Acrylamide Conc. %', 'Bis-acrylamide conc %', 'Photo-initiator conc. %', 'Layer Height. (micron)',
          'Bottom Layer exposure time (s) ', 'Exposure time (s)', 'Frequency (Hz)',
          'Storage modulus (Pa)', 'Loss modulus (Pa)']

rows = []

for i, j, k in zip(frequency, storage_modulus, loss_modulus):
    input_data = [[i, j, k]]
    output_parameters = generate_outputs(best_cvae, input_data, num_samples=1)[0]
    combined_data = list(output_parameters[0]) + [i, j, k]
    rows.append(combined_data)

df = pd.DataFrame(rows, columns=columns)
# Save the DataFrame to a CSV file
df.to_csv('synth_cvae_base.csv', index=False)
print("\nSaved synthetic data!")

[CVAE-Base] Epoch 1, Train Total: 3.6985, Val Total: 0.7962
[CVAE-Base] Epoch 2, Train Total: 0.6877, Val Total: 0.2309
[CVAE-Base] Epoch 3, Train Total: 0.4569, Val Total: 0.1915
[CVAE-Base] Epoch 4, Train Total: 0.4089, Val Total: 0.2506
[CVAE-Base] Epoch 5, Train Total: 0.3656, Val Total: 0.2360
[CVAE-Base] Epoch 6, Train Total: 0.3762, Val Total: 0.1964
[CVAE-Base] Epoch 7, Train Total: 0.3324, Val Total: 0.1300
[CVAE-Base] Epoch 8, Train Total: 0.3047, Val Total: 0.0946
[CVAE-Base] Epoch 9, Train Total: 0.2888, Val Total: 0.1213
[CVAE-Base] Epoch 10, Train Total: 0.3060, Val Total: 0.1112
[CVAE-Base] Epoch 11, Train Total: 0.2632, Val Total: 0.1429
[CVAE-Base] Epoch 12, Train Total: 0.2604, Val Total: 0.1240
[CVAE-Base] Epoch 13, Train Total: 0.2388, Val Total: 0.0855
[CVAE-Base] Epoch 14, Train Total: 0.2386, Val Total: 0.0789
[CVAE-Base] Epoch 15, Train Total: 0.2282, Val Total: 0.0819
[CVAE-Base] Epoch 16, Train Total: 0.2068, Val Total: 0.0724
[CVAE-Base] Epoch 17, Train Total